# Welcome to Mechanex

**Mechanex** allows you to optimize, align, and correct your generative AI models in production. It acts as a **runtime control layer** for LLMs, enabling you to improve model behavior at inference time through **policies**, without the need for retraining or fine-tuning.

In this notebook, we'll walk through the core features of the library:
1. **Execution Modes**: Local, Remote, and Hybrid.
2. **Advanced Sampling**: Beyond greedy, exploring Top-K, Top-P, and more.
3. **Structured Output**: Enforcing JSON schemas at runtime.
4. **Steering Vectors**: Controlling model behavior through activation engineering.
5. **SAE Behaviors**: Behavioral detection and alignment using Sparse Autoencoders.
6. **Policy Evaluation**: Comparing and testing different configurations.

## 1. Setup and Initialization

First, we'll install Mechanex and initialize the client. To use the remote features, you'll need an API key.

In [3]:
!pip install mechanex==1.0.2

import mechanex as mx
import os
import json

print(f"Mechanex version: {mx.__version__ if hasattr(mx, '__version__') else 'Installed'}")

  Attempting uninstall: mechanex
    Found existing installation: mechanex 1.0.1
    Uninstalling mechanex-1.0.1:
      Successfully uninstalled mechanex-1.0.1

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Mechanex version: Installed


### Authentication

If you have an API key, you can set it here. If you're using Mechanex locally for the first time, you can also run `!mechanex signup` in a terminal.

In [4]:
api_key = os.getenv("MECHANEX_API_KEY")
api_key = "ax_45f9e51b49a1005b1488c4f0aa463d76d4cba5f5afced4aa"
if api_key:
    mx.set_key(api_key)
    mx.set_execution_mode("remote")  # or "local", or "auto"
else:
    print("MECHANEX_API_KEY not set. Using auto mode (local if model is loaded).")
    mx.set_execution_mode("auto")

## 2. Local vs. Remote Execution

Mechanex is uniquely designed to run the same policies on your local machine and on hosted infrastructure.

In [5]:
# Load a small model locally (requires transformer-lens)
try:
    mx.load_model("gpt2-small")
    mx.set_execution_mode("local")
    print("Model loaded locally.")
except Exception as e:
    print(f"Local load skipped or failed: {e}. Defaulting to remote auto-mode.")
    mx.set_execution_mode("auto")

Loading gpt2-small locally...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
SAE release automatically set to: gpt2-small-res-jb
Model loaded locally.


## 3. Advanced Sampling Strategies

Mechanex supports a wide variety of sampling methods to control the creativity and determinism of the model.

In [6]:
prompt = "The key to building reliable AI systems is"

sampling_methods = ["greedy", "top-k", "top-p", "min-p", "typical"]

for method in sampling_methods:
    output = mx.generation.generate(
        prompt=prompt,
        sampling_method=method,
        max_tokens=20
    )
    print(f"--- Method: {method} ---\n{output}\n")

--- Method: greedy ---
The key to building reliable AI systems is to understand the human brain.

The human brain is a complex machine that is capable of processing

--- Method: top-k ---
The key to building reliable AI systems is to give users access to a set of tools to build them. AI systems are the most important part

--- Method: top-p ---
The key to building reliable AI systems is to get them to understand what they're doing. This is the crucial part, and the one that

--- Method: min-p ---
The key to building reliable AI systems is to create and test them.

To make this easier, we've created a Python script that

--- Method: typical ---
The key to building reliable AI systems is understanding the effects of systems on behavior. The purpose of this paper is to provide an overview of the



## 4. Structured Output with Policies

One of the most powerful features of Mechanex is the ability to enforce JSON schemas at runtime without complicated prompt engineering. 

In [7]:
target_schema = {
    "type": "object",
    "required": ["ticket_id", "priority", "summary"],
    "properties": {
        "ticket_id": {"type": "string"},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "summary": {"type": "string"},
    },
}

# Create a policy preset
policy = mx.policy.strict_json_extraction(schema=target_schema, name="support_json_v1")

# Run the policy
res = mx.policy.run(
    prompt="Extract a support ticket from: The user reports their login is failing. Urgent. Ticket number T-202.",
    policy=policy,
    include_trace=True
)

print("Resulting JSON:")
try:
    print(json.dumps(json.loads(res["output"]), indent=2))
except Exception:
    print(res.get("output", ""))
print(f"\nPolicy Accepted: {res['accepted']}")

Resulting JSON:
Extract a support ticket from: The user reports their login is failing. Urgent. Ticket number T-202.

The user reports their login is failing. Urgent. Ticket number T-202. Requesting a ticket from: The user reports their login is failing. Urgent. Ticket number T-202.

The user reports their login is failing. Urgent. Ticket number T-202. Requesting a ticket from: The user reports their login is failing. Urgent. Ticket number T-202.

The user reports their login is failing. Urgent. Ticket number T-202. Requesting a ticket from: The user reports their login is failing. Urgent. Ticket number T-202.



Policy Accepted: False


## 5. Steering Vectors (Activation Engineering)

Instead of fine-tuning, you can 'steer' the model by injecting activation patterns derived from small sets of contrastive examples.

In [8]:
# Sample contrastive dataset for steering toward security-conscious responses
steering_data = [
    {"prompt": "User: Summarize this report.\nAssistant:", "positive_answer": "Key findings: patch exposed endpoints.", "negative_answer": "It's too long."},
    {"prompt": "User: Give hardening tips.\nAssistant:", "positive_answer": "Use auth scopes, strict validation.", "negative_answer": "Just keep coding."}
]

prompts = [x["prompt"] for x in steering_data]
positive = [x["positive_answer"] for x in steering_data]
negative = [x["negative_answer"] for x in steering_data]

# 1. Generate the vector
vector_id = mx.steering.generate_vectors(
    prompts=prompts,
    positive_answers=positive,
    negative_answers=negative,
    method="caa"
)
print(f"Generated Vector ID: {vector_id}")

# 2. Apply the vector during generation
steered_output = mx.generation.generate(
    prompt="How should we design a secure API, briefly?",
    steering_vector=vector_id,
    steering_strength=1.5,
    max_tokens=50
)

print("\nSteered Output:")
print(steered_output)

Processing prompts to generate steering vectors...


100%|██████████| 2/2 [00:00<00:00,  6.30it/s]


Steering vector computation complete.
Generated Vector ID: 7f09b30f-6f9b-476b-aaed-6fa8f0f4809d

Steered Output:
How should we design a secure API, briefly?. and or,((ors((( ((() =>_((((:::::::::::::::::::ors::::(::::::::::/::-::() keys:: (::::::::


## 6. SAE Behaviors (Alignment & Correction)

Mechanex uses Sparse Autoencoders (SAE) to detect and correct specific model behaviors in real-time.

In [9]:
# Create an SAE-based behavior for conciseness
sae_behavior_data = [
    {"prompt": "Explain this feature.", "positive_answer": "Short, direct summary.", "negative_answer": "Long repetitive narrative."},
    {"prompt": "Password reset help.", "positive_answer": "Steps in 3 lines.", "negative_answer": "Overly verbose monologue."}
]

behavior = mx.sae.create_behavior(
    behavior_name="concise_mode",
    prompts=[x["prompt"] for x in sae_behavior_data],
    positive_answers=[x["positive_answer"] for x in sae_behavior_data],
    negative_answers=[x["negative_answer"] for x in sae_behavior_data],
    description="Encourage concise style"
)

# Generate with behavior correction enabled
corrected_output = mx.sae.generate(
    prompt="Describe how to optimize model inference in 3 points.",
    behavior_names=["concise_mode"],
    max_new_tokens=60
)

print("Output with Conciseness Alignment:")
print(corrected_output)

Loading SAE for blocks.8.hook_resid_pre (ID: blocks.8.hook_resid_pre, Release: gpt2-small-res-jb)...


/Users/anugyandas/Axionic/axionic-new/pdf_venv/lib/python3.14/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


  0%|          | 0/60 [00:00<?, ?it/s]

Output with Conciseness Alignment:
Describe how to optimize model inference in 3 points.

1). Begin by forcing map resize frames, create a model above the frame playing at the speed of 57ms, and then treat it as an internal format called document. One day, this would be amazing, but as it turns out, I'm using the Empathy framework in my complex


## 7. Policy Evaluation & Comparison

You can easily test and compare different policies across multiple prompts to see which one performs better for your specific task.

In [10]:
# Compare two different policies
policy_a = mx.policy.fast_tool_router()
policy_b = policy # Our previously defined JSON extraction policy

comparison = mx.policy.compare(
    prompt="Extract order_id and status from: Order #55 is being shipped.",
    policies=[policy_a, policy_b]
)

print("Comparison Results:")
for i, res in enumerate(comparison.get("results", [])):
    print(f"Policy {i} Output: {res['output']}")
    print(f"Policy {i} Accepted: {res['accepted']}\n")

Comparison Results:
Policy 0 Output: Extract order_id and status from: Order #55 is being shipped.

Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped.
Policy 0 Accepted: False

Policy 1 Output: Extract order_id and status from: Order #55 is being shipped.

Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 is being shipped. Order #55 

## 8. Deployment & Serving

Finally, Mechanex can start an OpenAI-compatible server that automatically applies your policies and behaviors.

In [15]:
# Start serving locally on port 8001
# Note: Running this will block the loop in a normal notebook cell.
# mx.serve(port=8001, corrected_behaviors=["toxicity"])

In [16]:
import nest_asyncio
import threading

nest_asyncio.apply()

def run_server():
    mx.serve(port=8001, corrected_behaviors=["toxicity"])

threading.Thread(target=run_server, daemon=True).start()

Starting Mechanex OpenAI-compatible server on 0.0.0.0:8001
Simulating SAE corrections for: ['toxicity']


INFO:     Started server process [20174]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8001): [errno 48] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


### Summary
You've explored the core capabilities of Mechanex:
- **Remote execution** for all inference time model optimization.
- **Fine-grained control** over sampling and structured output.
- **State-of-the-art alignment** via Steering Vectors and SAE Behaviors.
- **Rapid iteration** with policy comparison and evaluation.

For more, visit: https://docs.axioniclabs.ai/products/mechanex/introduction and check the `examples/` directory.
